# Dimension indexes on the existing CCI baseline

Two configurations: CCI baseline and all nine dimension indexes together. Facts and query SQL stay unchanged. Positive time change means slower.

Run the experiment first using the adjacent README. This notebook validates raw measurements and saved outputs before producing tables and figures. It does not connect to SQL Server or run queries.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Works from the repository root or this notebook's directory.
HERE = next((p for p in [Path.cwd(), Path.cwd() / 'experiments/dimension_indexes']
             if (p / 'analyze.py').is_file() and (p / 'indexes.sql').is_file()), None)
if HERE is None:
    raise RuntimeError('Open Jupyter from the repository root or experiments/dimension_indexes.')
HERE = HERE.resolve()
ROOT = HERE.parents[1]
spec = importlib.util.spec_from_file_location('dimension_analysis', HERE / 'analyze.py')
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
RUNNER_SPEC = importlib.util.spec_from_file_location('dimension_runner', HERE / 'run.py')
runner = importlib.util.module_from_spec(RUNNER_SPEC)
RUNNER_SPEC.loader.exec_module(runner)
SCENARIOS = list(analysis.SCENARIOS)
INDEXES = [index for group in runner.INDEXES.values() for index in group]
EXPORTS = HERE / 'exports'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})


## Select and validate runs

Leave `RUN_DIRS` empty to discover completed runs. If several runs exist for one scale, select their paths explicitly. Failed and unfinished runs are excluded. Relative paths are resolved from the repository root. No synthetic or historical results are substituted.

In [ ]:
RUN_DIRS = []
# Example:
# RUN_DIRS = ['benchmark_results/dimension_indexes/1GB_TIMESTAMP_ID']

if RUN_DIRS:
    folders = [Path(p) if Path(p).is_absolute() else ROOT / p for p in RUN_DIRS]
else:
    folders = sorted(p for p in (ROOT / 'benchmark_results/dimension_indexes').glob('*')
                     if (p / 'STATUS').is_file() and (p / 'STATUS').read_text().strip() == 'COMPLETE')
    scales = [json.loads((p / 'manifest.json').read_text())['scale'] for p in folders]
    if len(scales) != len(set(scales)):
        raise ValueError('Multiple completed runs for one scale: set RUN_DIRS explicitly.')
if not folders:
    raise RuntimeError('No completed dimension experiment runs. Follow README.md, then rerun this notebook.')
results = [analysis.analyze(p) for p in folders]
EXPORTS.mkdir(exist_ok=True)

def table(key):
    frames = []
    for result in results:
        frame = pd.DataFrame(result[key])
        frame['run'] = Path(result['folder']).name
        frame['scale'] = result['manifest']['scale']
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

summary, queries, passes, checks = [table(k) for k in ['summary', 'queries', 'passes', 'output_checks']]
index_review = table('index_review')
index_review.to_csv(EXPORTS / 'index_review.csv', index=False)
for name, frame in [('scenario_summary', summary), ('query_comparisons', queries),
                    ('workload_passes', passes), ('output_checks', checks)]:
    frame.to_csv(EXPORTS / (name + '.csv'), index=False)
provenance = pd.DataFrame([dict(run=Path(r['folder']).name, scale=r['manifest']['scale'],
    source=r['manifest']['source_database'], queries=len(r['manifest']['queries']),
    passes=r['manifest']['passes'], seed=r['manifest']['seed'],
    order=' -> '.join(r['manifest']['scenario_order'])) for r in results])
display(provenance)
provenance.to_csv(EXPORTS / 'provenance.csv', index=False)
reference = results[0]['manifest']['queries']
if any(r['manifest']['queries'] != reference for r in results[1:]):
    display(Markdown('**SQL differs between selected runs.** Within-run comparisons remain usable; differences across sizes cannot be attributed solely to data growth.'))
if any(r['manifest']['files']['indexes.sql'] != results[0]['manifest']['files']['indexes.sql'] for r in results[1:]):
    raise ValueError('Index definitions differ between runs; select matching experiments.')
reviews = checks[checks.status == 'REVIEW_REQUIRED']
if not reviews.empty:
    display(Markdown('**Output differences require review before accepting performance conclusions.**'))
    display(reviews)
else:
    display(Markdown('No unresolved differences in the saved warm-up outputs. This is a rendered-output check, not a formal SQL-equivalence proof.'))


## Total workload and added index space

Total time is the sum of mean query durations, equivalent to the mean total of complete measured passes. Error bars show sample SD between workload passes, not confidence intervals. Index space is additional space above the shared CCI baseline. Setup time excludes statistics and plan capture.

In [ ]:
display(summary[['run', 'scenario', 'mean_s', 'sd_s', 'change_pct', 'speedup',
                 'added_mib', 'added_storage_pct', 'setup_s', 'output_reviews']].round(3))

def savefig(fig, name):
    fig.tight_layout()
    for extension in ['png', 'svg']:
        fig.savefig(EXPORTS / f'{name}.{extension}', bbox_inches='tight')
    plt.show()

for result in results:
    run = Path(result['folder']).name
    data = summary[summary.run == run].set_index('scenario').reindex(SCENARIOS)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].barh(SCENARIOS, data.mean_s, xerr=data.sd_s, capsize=3, color='#268795')
    axes[0].set(xlabel='Workload time (s), mean +/- SD', title=f'{run}: elapsed time')
    axes[1].barh(SCENARIOS, data.added_mib, color='#c57e44')
    axes[1].set(xlabel='Additional index storage (MiB)', title='Space above CCI baseline')
    for ax in axes:
        ax.invert_yaxis()
    savefig(fig, run + '_01_time_storage')


## Stability between measured passes

Scenario order is shuffled once using the recorded seed. Pass numbers are local to each scenario: equal numbers do not represent simultaneous or randomized matched pairs. Repeat with another seed to check order sensitivity. Docker/sqlcmd startup and output transfer are included in every timing; data buffers remain warm.

In [ ]:
for result in results:
    run = Path(result['folder']).name
    data = passes[passes.run == run]
    fig, ax = plt.subplots(figsize=(10, 4))
    for scenario in SCENARIOS:
        part = data[data.scenario == scenario]
        ax.plot(part.pass_number, part.total_s, marker='o', label=scenario)
    ax.set(xlabel='Measured pass', ylabel='Total workload time (s)', title=run)
    ax.set_xticks(sorted(data.pass_number.unique()))
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    savefig(fig, run + '_02_stability')


## Improvements and regressions by query

The 5% descriptive tolerance below is adjustable and is **not a significance test**. Counts weight queries equally; total workload time weights their durations. The contribution plot shows the 15 largest absolute query changes for all dimensions together; the residual includes every remaining query, so the bars sum to the exact workload change.

In [ ]:
TOLERANCE_PCT = 5
candidate = queries[queries.scenario != 'baseline'].copy()
candidate['classification'] = candidate.change_pct.map(
    lambda x: 'Faster' if x < -TOLERANCE_PCT else ('Slower' if x > TOLERANCE_PCT else 'Within tolerance'))
counts = candidate.groupby(['run', 'scenario', 'classification']).size().unstack(fill_value=0)
display(counts)
counts.to_csv(EXPORTS / 'query_counts.csv')
display(candidate.sort_values('delta_s', ascending=False)[
    ['run', 'scenario', 'query_id', 'baseline_ms', 'mean_ms', 'sd_ms', 'delta_s', 'change_pct']].head(20).round(3))
for result in results:
    run = Path(result['folder']).name
    part = queries[(queries.run == run) & (queries.scenario == 'all_dimensions')].set_index('query_id')
    top = part.delta_s.abs().nlargest(15).index
    values = part.loc[top, 'delta_s'].copy()
    values.loc['All other queries'] = part.drop(top).delta_s.sum()
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(values.index, values.values, color=['#bd604b' if v > 0 else '#268795' for v in values])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    ax.set(xlabel='Seconds added (+) or removed (-) per workload pass', title=run + ': all dimensions vs baseline')
    savefig(fig, run + '_03_query_contributions')


## Does the optimizer select the new indexes?

These are distinct-query references in **estimated** plans, not executed seeks, actual reads or execution counts. Zero-use indexes remain visible. A referenced index is not proof of a faster plan.

The review table lists every index, its space, distinct-query count and query identifiers. `not_selected` indexes are candidates for removal for this workload. Selection alone does not establish benefit. Review the table across all measured scales before discarding an index; repeat the benchmark after removal.

In [ ]:
display(index_review[['run', 'index', 'table', 'columns', 'added_mib',
                      'estimated_query_count', 'queries', 'selection', 'review_action']])
candidates = index_review[index_review.selection == 'not_selected']
if candidates.empty:
    display(Markdown('Every index is referenced in at least one estimated plan in each selected run.'))
else:
    display(Markdown('**Candidates for removal in each run (no estimated-plan references):**'))
    display(candidates[['run', 'index', 'added_mib', 'review_action']])
    candidates.to_csv(EXPORTS / 'removal_candidates.csv', index=False)
for result in results:
    run = Path(result['folder']).name
    part = index_review[index_review.run == run]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(part['index'], part.estimated_query_count, color='#687eaa')
    ax.invert_yaxis()
    ax.set(xlabel='Distinct queries referencing the index in estimated plans', title=run)
    savefig(fig, run + '_04_index_selection')


## Observed results for the report

The following text is calculated from validated runs. Five passes, estimated plans, warm buffers and client overhead limit interpretation. There are no write/maintenance measurements. A small observed mean difference is not a statistically established effect.

In [ ]:
for result in results:
    run = Path(result['folder']).name
    data = summary[summary.run == run]
    text = [f'### {run}']
    for row in data.itertuples():
        text.append(f'- **{row.scenario}:** {row.mean_s:.3f} s (SD {row.sd_s:.3f} s), '
                    f'{row.change_pct:+.2f}% vs baseline, {row.added_mib:.2f} MiB added; '
                    f'{row.output_reviews} output reviews.')
    if data.output_reviews.sum():
        text.append('**Output reviews remain unresolved; performance conclusions are provisional.**')
    else:
        winner = data.loc[data.mean_s.idxmin()]
        text.append(f'Lowest observed mean: **{winner.scenario}**. This does not establish statistical significance.')
    report = '\n\n'.join(text)
    (EXPORTS / (run + '_observed_results.md')).write_text(report, encoding='utf-8')
    display(Markdown(report))
print('Exports:', EXPORTS)
